# NER Inference on TEST SET — NB1b (BioBERT) + NB1b_pubmed (PubMedBERT)

Runs the full inference pipeline on `articles_test.json` for the **two best models only**.

| Tag | Model folder | Prediction file |
|-----|-------------|----------------|
| NB1b | `bert_biomedbert_ner_twopass_gold_silver_bronze` | `pred_NB1b_gold_silver_bronze_TEST.json` |
| NB1b_pubmed | `pubmedbert_ner_twopass_gold_silver_bronze` | `pred_NB1b_pubmed_gold_silver_bronze_TEST.json` |

**No evaluation step** — the test set has no ground-truth annotations.

Output files feed directly into `bert_NER_ensemble_TEST.ipynb`.

## 0. Imports & reproducibility

In [1]:
import json
import re
import copy
import numpy as np
import torch
from pathlib import Path
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForTokenClassification

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

C:\Users\super\Documents\UniPd\ATA\GutBrainIE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.11.0.dev20260204+cu128
CUDA available: True
Device: cuda


## 1. Label space (must match training)

In [2]:
ENTITY_LABELS = [
    "anatomical location",
    "animal",
    "bacteria",
    "biomedical technique",
    "chemical",
    "DDF",
    "dietary supplement",
    "drug",
    "food",
    "gene",
    "human",
    "microbiome",
    "statistical technique",
]

label_list = ["O"]
for lab in ENTITY_LABELS:
    label_list.append(f"B-{lab}")
    label_list.append(f"I-{lab}")

label2id = {k: v for v, k in enumerate(label_list)}
id2label  = {v: k for k, v in label2id.items()}

print(f"Total labels: {len(label_list)}")  # expected: 27

Total labels: 27


## 2. Paths

In [3]:
def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "data").exists() and (p / "src").exists():
            return p
    raise FileNotFoundError(
        f"Cannot find repo root (needs 'data' and 'src' subfolders). "
        f"Started from: {start}"
    )

PROJECT_ROOT = find_repo_root(Path.cwd())
MODELS_DIR   = PROJECT_ROOT / "src" / "ner" / "models"
PRED_DIR     = PROJECT_ROOT / "src" / "ner" / "predictions"/ "test_set"
PRED_DIR.mkdir(parents=True, exist_ok=True)

# ── TEST SET: articles only, no annotations ───────────────────────────────────
TEST_PATH = (
    PROJECT_ROOT
    / "data" / "GutBrainIE_Full_Collection_2026"
    / "Articles" / "json_format" / "articles_test.json"
)

# ── Only the two best models ──────────────────────────────────────────────────
MODELS = {
    "NB1b":        MODELS_DIR / "bert_biomedbert_ner_twopass_gold_silver_bronze",
    "NB1b_pubmed": MODELS_DIR / "pubmedbert_ner_twopass_gold_silver_bronze",
}

PRED_FILES = {
    "NB1b":        PRED_DIR / "pred_NB1b_gold_silver_bronze_TEST.json",
    "NB1b_pubmed": PRED_DIR / "pred_NB1b_pubmed_gold_silver_bronze_TEST.json",
}

print("Project root :", PROJECT_ROOT)
print("Test file    :", TEST_PATH, "| exists:", TEST_PATH.exists())
print()
for tag, p in MODELS.items():
    status = "✓" if p.exists() else "✗ MISSING"
    print(f"  {status}  {tag:12s}  {p.name}")

Project root : C:\Users\super\Documents\UniPd\ATA\GutBrainIE
Test file    : C:\Users\super\Documents\UniPd\ATA\GutBrainIE\data\GutBrainIE_Full_Collection_2026\Articles\json_format\articles_test.json | exists: True

  ✓  NB1b          bert_biomedbert_ner_twopass_gold_silver_bronze
  ✓  NB1b_pubmed   pubmedbert_ner_twopass_gold_silver_bronze


## 3. Load test data

`articles_test.json` has **no `entities` field** — only `metadata` with `title` and `abstract`.

In [4]:
with TEST_PATH.open(encoding="utf-8") as f:
    test_data = json.load(f)

def prepare_documents_test(data: dict) -> list:
    """Split each article into title + abstract segments. No entities field expected."""
    docs = []
    for pmid, article in data.items():
        # articles_test.json may have metadata nested or flat — handle both
        meta = article.get("metadata", article)
        for loc in ("title", "abstract"):
            text = (meta.get(loc) or "").strip()
            if not text:
                continue
            docs.append({
                "pmid":     str(pmid),
                "location": loc,
                "text":     text,
            })
    return docs

test_documents = prepare_documents_test(test_data)
print(f"Test articles : {len(test_data)}")
print(f"Test segments : {len(test_documents)}  (title + abstract)")

Test articles : 80
Test segments : 160  (title + abstract)


## 4. Inference pipeline (identical to dev version)

In [5]:
# ── Thresholds (tuned on dev) ─────────────────────────────────────────────────
LABEL_THRESH_HIGH = {
    "DDF":                    0.88,
    "bacteria":               0.84,
    "statistical technique":  0.91,
    "biomedical technique":   0.78,
    "gene":                   0.68,
    "food":                   0.60,
    "chemical":               0.72,
    "dietary supplement":     0.78,
    "drug":                   0.80,
    "microbiome":             0.78,
    "anatomical location":    0.78,
    "human":                  0.70,
    "animal":                 0.70,
}
LABEL_THRESH_RECALL = {
    "food":               0.43,
    "chemical":           0.65,
    "bacteria":           0.80,
    "dietary supplement": 0.72,
}
RECALL_LABELS  = {"chemical", "food", "bacteria", "dietary supplement"}
DEFAULT_THRESH = 0.80

print("Thresholds loaded.")

Thresholds loaded.


In [6]:
# ── FP filter lists ───────────────────────────────────────────────────────────
BAD_BACTERIA   = {"bacteria", "micro", "microbes", "microorganisms", "genera", "taxa"}
BAD_CHEMICAL   = {"metabolites", "neurotransmitters"}
BAD_DIETSUPP   = {"nnss", "sp", "fep", "ns9", "pro"}
BAD_MICROBIOME = {"micro", "microbiota", "gut"}
DIET_CONCEPT   = {
    "diet", "ketogenic diet", "high-fat diet", "high fat diet",
    "high glycemic diet", "vegetarian diet", "balanced diet",
    "western diet", "mediterranean diet",
}
BAD_FOOD_EXACT = {
    "control", "ketogenic", "high-fat", "high fat", "high",
    "glycemic index", "lycemic index", "food", "ingested food",
}
GENE_LIKE = re.compile(
    r"^(il-\d+|tnf(-?α)?|ifn(-?γ)?|tgf(-?β\d*)?|snca|park7|dj-1|mapt|apoe\d*|hla-[a-z0-9\*\:]+)$",
    re.IGNORECASE,
)
CHEM_LIKE = re.compile(
    r"(aβ|amyloid|scfa|gaba|succinate|butyrate|propionate|acetate|\b[a-z]+ate\b|\b[a-z]+acid\b|\(\d+\-\d+\))",
    re.IGNORECASE,
)
FOOD_ANCHORS = {
    "kefir", "yogurt", "milk", "cheese", "cookie",
    "lentil", "lentils", "buckwheat", "wheat", "rice", "tea", "coffee",
}
SUPP_HARD = {"capsule", "tablet", "extract", "powder"}
SUPP_SOFT = {"probiotic", "probiotics", "prebiotic", "prebiotics",
             "synbiotic", "synbiotics", "supplement"}
TRIM_CHARS = " \t\n\r.,;:()[]{}<>\"'"

print("Filter lists loaded.")

Filter lists loaded.


In [7]:
# ── Utility functions ─────────────────────────────────────────────────────────

def normalize_span(s: str) -> str:
    s = (s or "").strip().lower()
    return re.sub(r"\s+", " ", s)


def trim_entity_span(e: dict, text: str) -> dict:
    s   = max(0, min(int(e["start_idx"]), len(text)))
    end = max(0, min(int(e["end_idx"]),   len(text) - 1))
    while s <= end and text[s]   in TRIM_CHARS: s   += 1
    while end >= s and text[end] in TRIM_CHARS: end -= 1
    if s <= end:
        e["start_idx"] = s
        e["end_idx"]   = end
        e["text_span"] = text[s : end + 1]
    return e


def apply_simple_filters(entities: list) -> list:
    out = []
    for e in entities:
        s   = normalize_span(e.get("text_span", ""))
        lab = e["label"]
        if not s or len(s) <= 1:                                     continue
        if "<" in s or ">" in s:                                     continue
        if lab == "bacteria"          and s in BAD_BACTERIA:         continue
        if lab == "dietary supplement" and s in BAD_DIETSUPP:        continue
        if lab == "chemical"          and s in BAD_CHEMICAL:         continue
        if lab == "microbiome"        and s in BAD_MICROBIOME:       continue
        if lab == "food":
            if s in DIET_CONCEPT or s.endswith(" diet"): continue
            if s in BAD_FOOD_EXACT:                       continue
        out.append(e)
    return out


def postprocess_gene_vs_chemical(entities: list) -> list:
    for e in entities:
        s = normalize_span(e.get("text_span", ""))
        if e["label"] == "chemical" and GENE_LIKE.match(s):
            e["label"] = "gene"
        elif e["label"] == "gene" and CHEM_LIKE.search(s):
            e["label"] = "chemical"
    return entities


def postprocess_food_vs_supp(entities: list) -> list:
    for e in entities:
        s = normalize_span(e.get("text_span", ""))
        if any(w in s for w in FOOD_ANCHORS):
            if e["label"] in {"dietary supplement", "food"}: e["label"] = "food"
        elif any(w in s for w in SUPP_HARD):
            if e["label"] in {"dietary supplement", "food"}: e["label"] = "dietary supplement"
        elif e["label"] == "food" and any(w in s for w in SUPP_SOFT):
            e["label"] = "dietary supplement"
    return entities


def passes_threshold(e: dict, label_thresh: dict) -> bool:
    thr = label_thresh.get(e["label"], DEFAULT_THRESH)
    if e["label"] == "food" and "diet" in normalize_span(e.get("text_span", "")):
        thr = max(thr, 0.70)
    return e.get("score", 0.0) >= thr


def filter_by_threshold(entities: list, label_thresh: dict) -> list:
    return [e for e in entities if passes_threshold(e, label_thresh)]


def span_iou(a: dict, b: dict) -> float:
    inter = max(0, min(a["end_idx"], b["end_idx"]) - max(a["start_idx"], b["start_idx"]) + 1)
    if inter == 0: return 0.0
    la = a["end_idx"] - a["start_idx"] + 1
    lb = b["end_idx"] - b["start_idx"] + 1
    return inter / (la + lb - inter)


def dedup_segment(entities: list) -> list:
    seen, out = set(), []
    for e in entities:
        k = (e["start_idx"], e["end_idx"], e["location"])
        if k not in seen:
            seen.add(k)
            out.append(e)
    return out


def soft_overlap_prune(entities: list, same_label_only: bool = True, ratio_thr: float = 0.85) -> list:
    if not entities:
        return entities
    by_loc = {}
    for e in entities:
        by_loc.setdefault(e["location"], []).append(e)
    result = []
    for loc, ents in by_loc.items():
        ents = sorted(ents, key=lambda x: x["start_idx"])
        keep = [True] * len(ents)
        for i in range(len(ents)):
            if not keep[i]: continue
            for j in range(i + 1, len(ents)):
                if not keep[j]: continue
                if same_label_only and ents[i]["label"] != ents[j]["label"]: continue
                iou = span_iou(ents[i], ents[j])
                if iou >= ratio_thr:
                    li = ents[i]["end_idx"] - ents[i]["start_idx"]
                    lj = ents[j]["end_idx"] - ents[j]["start_idx"]
                    if lj > li: keep[i] = False
                    else:       keep[j] = False
        result.extend(e for e, k in zip(ents, keep) if k)
    return result


def merge_two_pass(ents_high: list, ents_rec: list, recall_labels: set) -> list:
    covered = {(e["start_idx"], e["end_idx"], e["location"]) for e in ents_high}
    extras  = [
        e for e in ents_rec
        if e["label"] in recall_labels
        and (e["start_idx"], e["end_idx"], e["location"]) not in covered
    ]
    return ents_high + extras


print("Utility functions defined.")

Utility functions defined.


In [8]:
# ── BIO decoder ───────────────────────────────────────────────────────────────

def predict_entities_with_scores(model, tokenizer, text: str, id2label, label2id, device):
    encoding = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        return_offsets_mapping=True,
        padding=False,
    )
    offset_mapping = encoding.pop("offset_mapping")[0].tolist()
    encoding = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        logits = model(**encoding).logits[0]
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    preds = np.argmax(probs, axis=-1)

    entities, current = [], None

    def _start(lbl, s, e, tp):
        return {"label": lbl, "start_idx": s, "end_idx": e, "_tp": [tp]}

    def _extend(ent, e, tp):
        ent["end_idx"] = e
        ent["_tp"].append(tp)

    for idx, (tag_id, (s, e)) in enumerate(zip(preds, offset_mapping)):
        if s == e == 0: continue  # special tokens
        tag  = id2label[tag_id]
        prob = float(probs[idx, tag_id])

        if tag == "O":
            if current: entities.append(current); current = None
        elif tag.startswith("B-"):
            if current: entities.append(current)
            current = _start(tag[2:], s, e, prob)
        elif tag.startswith("I-"):
            lbl = tag[2:]
            if current is None:           current = _start(lbl, s, e, prob)
            elif lbl != current["label"]: entities.append(current); current = _start(lbl, s, e, prob)
            else:                         _extend(current, e, prob)
        else:
            if current: entities.append(current); current = None

    if current: entities.append(current)

    for ent in entities:
        tp = ent.pop("_tp", [])
        ent["score"]     = float(np.median(tp)) if tp else 0.0
        ent["text_span"] = text[ent["start_idx"]: ent["end_idx"] + 1]

    return entities


def predict_segment_two_pass(model, tokenizer, text: str, location: str, device) -> list:
    """Full two-pass inference for one title or abstract segment."""
    ents_raw = predict_entities_with_scores(model, tokenizer, text, id2label, label2id, device)

    trimmed = []
    for e in ents_raw:
        e = trim_entity_span(e, text)
        if e and e.get("text_span"):
            e["location"] = location
            trimmed.append(e)

    trimmed = postprocess_gene_vs_chemical(trimmed)
    trimmed = postprocess_food_vs_supp(trimmed)

    ents_high = filter_by_threshold(trimmed, LABEL_THRESH_HIGH)
    ents_high = apply_simple_filters(ents_high)

    ents_rec  = filter_by_threshold(trimmed, LABEL_THRESH_RECALL)
    ents_rec  = apply_simple_filters(ents_rec)

    merged = merge_two_pass(ents_high, ents_rec, RECALL_LABELS)
    merged = dedup_segment(merged)
    merged = soft_overlap_prune(merged, same_label_only=True, ratio_thr=0.85)

    # Remove score — not part of submission format
    for e in merged:
        e.pop("score", None)

    return merged


print("Inference functions defined.")

Inference functions defined.


## 5. Run inference — NB1b + NB1b_pubmed on TEST SET

No evaluation is performed (no ground truth available).

In [9]:
for tag, model_path in MODELS.items():
    print(f"\n{'='*60}")
    print(f"  Model: {tag}  —  {model_path.name}")
    print(f"{'='*60}")

    if not model_path.exists():
        print(f"  ✗ SKIPPED — folder not found: {model_path}")
        continue

    print("  Loading model…")
    tokenizer = AutoTokenizer.from_pretrained(str(model_path))
    model = AutoModelForTokenClassification.from_pretrained(
        str(model_path),
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
    ).to(DEVICE)
    model.eval()

    predictions = {}
    for doc in tqdm(test_documents, desc=f"  Inferring {tag}", leave=False):
        pmid     = doc["pmid"]
        location = doc["location"]
        text     = doc["text"]
        ents = predict_segment_two_pass(model, tokenizer, text, location, DEVICE)
        predictions.setdefault(pmid, {"entities": []})
        predictions[pmid]["entities"].extend(ents)

    total_pred = sum(len(v["entities"]) for v in predictions.values())
    print(f"  ✓ Total predicted entities: {total_pred}")

    out_path = PRED_FILES[tag]
    with out_path.open("w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)
    print(f"  ✓ Saved: {out_path}")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n✓ Both models processed. Ready for ensemble.")


  Model: NB1b  —  bert_biomedbert_ner_twopass_gold_silver_bronze
  Loading model…


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2913.59it/s]


  ✓ Total predicted entities: 2554
  ✓ Saved: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\ner\predictions\test_set\pred_NB1b_gold_silver_bronze_TEST.json

  Model: NB1b_pubmed  —  pubmedbert_ner_twopass_gold_silver_bronze
  Loading model…


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3682.15it/s]
                                                                          

  ✓ Total predicted entities: 2422
  ✓ Saved: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\ner\predictions\test_set\pred_NB1b_pubmed_gold_silver_bronze_TEST.json

✓ Both models processed. Ready for ensemble.
